# Bruker / ScanImage structural volume alignment QC

This notebook runs the `slap2_volume_align` ScanImage alignment pipeline, then inspects the averaged output TIFF(s) and alignment diagnostics.

It is designed for large structural volume TIFFs where the file should be processed plane-by-plane rather than loaded into memory all at once. Your current reference stack appears to be organized as `slice_blocks`:

```text
z0 repeat0..19,
z1 repeat0..19,
z2 repeat0..19,
...
```

For the first full run, use a small plane range first, inspect the QC, then run the full stack.


## 1. Imports and repo setup

Run this notebook from either the repo root or the `notebooks/` directory. The cell below adds `src/` to `sys.path` so the package is importable before/without installation.


In [ ]:

from __future__ import annotations

import json
import math
import os
import sys
import warnings
from pathlib import Path
from typing import Optional

from IPython.display import display, HTML

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt


def find_repo_root(start: Optional[Path] = None) -> Path:
    """Find repo root from nested notebook directories."""
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "slap2_volume_align").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing pyproject.toml and src/slap2_volume_align. "
        f"Started from: {start}"
    )


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from slap2_volume_align.readers.tiff import (
    read_tiff_stack_spec,
    read_plane_channel_frames,
)
from slap2_volume_align.sources.scanimage.metadata import parse_scanimage_description
from slap2_volume_align.core.registration import (
    estimate_rigid_shift,
    make_template,
    mean_shifted_frames,
)
from slap2_volume_align.sources.scanimage.pipeline import (
    ScanImageAverageConfig,
    average_scanimage_volume,
)
from slap2_volume_align.core.straightening import (
    straighten_volume_tiff,
    estimate_z_straightening_shifts,
    apply_z_straightening,
    write_straightening_csv,
    make_straightening_qc_png,
)
from slap2_volume_align.core.bidirectional import apply_bidirectional_phase

print(f"Repo root: {REPO_ROOT}")
print(f"Source dir: {SRC_DIR}")
print(f"Python: {sys.version}")

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")
print("Imports OK")


## 2. User parameters

Edit these paths and parameters for the session you want to process.

Notes:

- `N_PLANES=177`, `REPEATS_PER_PLANE=20`, `N_CHANNELS=1`, and `ORDER='slice_blocks'` match the first test stack.
- For future two-channel ScanImage stacks, use `N_CHANNELS=2` and keep `ALIGNMENT_CHANNEL=0` if green/channel 1 should drive alignment.
- `OUTPUT_DTYPE='float32'` is recommended because averaging and subpixel shifts produce non-integer values.


In [ ]:
# --- Input/output paths ---
INPUT_TIF = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\836174\836174_2026-06-17_12-51-17\reference_volume\2026-06-17_836174_stack_00001.tif")
OUT_DIR = INPUT_TIF.parent / "avg_scanimage_python"

# --- Stack layout ---
N_PLANES = 177
REPEATS_PER_PLANE = 20
N_CHANNELS = 1
ALIGNMENT_CHANNEL = 0  # zero-based; 0 == channel 1
ORDER = "slice_blocks"  # "slice_blocks" or "volume_interleaved"

# --- Processing range ---
# Start with a smoke test, e.g. 0:3. Then set PLANE_STOP = None for the full volume.
PLANE_START = 0
PLANE_STOP = 3  # exclusive; set to None for full stack

# --- Registration parameters ---
ALIGN = True
REGISTRATION_BINNING = 8      # estimate shifts on downsampled images; shifts are applied full-res
UPSAMPLE_FACTOR = 3           # subpixel precision of phase correlation
MAX_SHIFT_PX = 20.0           # reject large/implausible shifts
HIGHPASS_SIGMA_PX = 8.0       # broad background subtraction before registration
INTERPOLATION_ORDER = 1       # 1 = bilinear; 0 = nearest; 3 = cubic but slower
TEMPLATE_METHOD = "median"    # "median" or "mean"

# --- Output options ---
OUTPUT_DTYPE = "float32"
OUTPUT_COMPRESSION = None     # None is fastest for validation; use "zlib" later if desired
WRITE_QC_PNG = True

# --- Notebook diagnostics ---
DISPLAY_DOWNSAMPLE = 8        # display only; saved TIFF remains full resolution
SELECTED_Z_FOR_RAW_CHECK = 0
Z_PROFILE_STEP = 1            # use 1 for all output planes; increase for faster coarse QC

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(INPUT_TIF)
print(OUT_DIR)


# --- Optional bidirectional line-phase correction ---
# Use BIDIPHASE=1 only if raw frames show odd/even line phase artifacts.
BIDIPHASE = 0
BIDI_LINE_PARITY = "odd"
BIDI_FILL_MODE = "preserve"

# --- Optional second-pass z-stack straightening ---
# This corrects residual lateral jitter between averaged z-planes.
STRAIGHTEN_Z = False
Z_ANCHOR = None                 # auto-pick high-contrast plane; or set an integer z index
Z_TEMPLATE_RADIUS = 2
Z_REGISTRATION_BINNING = 4
Z_UPSAMPLE_FACTOR = 10
Z_MAX_STEP_SHIFT_PX = 12.0
Z_HIGHPASS_SIGMA_PX = 16.0
Z_SMOOTH_WINDOW = 9


## 3. Inspect the source TIFF without loading it into memory

This confirms page count, shape, dtype, and a few ScanImage per-frame metadata fields.


In [ ]:

with tifffile.TiffFile(INPUT_TIF) as tif:
    print(f"n_pages: {len(tif.pages)}")
    print(f"page[0].shape: {tif.pages[0].shape}")
    print(f"page[0].dtype: {tif.pages[0].dtype}")
    print("\nFirst page description excerpt:")
    print((tif.pages[0].description or "")[:2000])

    probe_pages = sorted(set([
        0, 1, REPEATS_PER_PLANE - 1, REPEATS_PER_PLANE,
        2 * REPEATS_PER_PLANE - 1, 2 * REPEATS_PER_PLANE,
        len(tif.pages) - 1,
    ]))
    rows = []
    for p in probe_pages:
        desc = parse_scanimage_description(tif.pages[p].description)
        rows.append({
            "page": p,
            "frame_number": desc.frame_number,
            "acquisition_number": desc.acquisition_number,
            "frame_number_acquisition": desc.frame_number_acquisition,
            "frame_timestamp_sec": desc.frame_timestamp_sec,
        })

pd.DataFrame(rows)


## 4. Verify inferred layout

`read_tiff_stack_spec` checks that the requested layout accounts for every TIFF page. If `infer_from_descriptions=True`, it can also infer `n_planes` and `repeats_per_plane` from ScanImage fields like `acquisitionNumbers` and `frameNumberAcquisition`.


In [ ]:

spec = read_tiff_stack_spec(
    INPUT_TIF,
    n_planes=N_PLANES,
    repeats_per_plane=REPEATS_PER_PLANE,
    n_channels=N_CHANNELS,
    order=ORDER,
    infer_from_descriptions=True,
)

spec.to_dict()


## 5. Raw page-order/contact-sheet QC

This plots raw repeats for selected z-planes directly from the input TIFF. Adjacent tiles should look like noisy/motion-shifted views of the same z-plane if `ORDER='slice_blocks'` is correct.


In [ ]:

def robust_limits(img, pct=(1, 99.8)):
    finite = np.isfinite(img)
    if not finite.any():
        return 0, 1
    lo, hi = np.percentile(img[finite], pct)
    if hi <= lo:
        hi = lo + 1
    return lo, hi


def show_raw_repeats(
    input_tif: Path,
    z_indices=(0, 1, 2),
    channel_index=0,
    max_repeats=8,
    downsample=8,
):
    nrows = len(z_indices)
    ncols = min(max_repeats, spec.repeats_per_plane)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(2.2 * ncols, 2.2 * nrows), squeeze=False
    )

    with tifffile.TiffFile(input_tif) as tif:
        for row, z in enumerate(z_indices):
            frames = read_plane_channel_frames(
                tif, z_index=z, channel_index=channel_index, spec=spec
            )
            for col, frame in enumerate(frames[:ncols]):
                ax = axes[row, col]
                thumb = frame[::downsample, ::downsample]
                lo, hi = robust_limits(thumb)
                ax.imshow(thumb, cmap="gray", vmin=lo, vmax=hi)
                ax.set_title(f"z={z}, r={col}", fontsize=8)
                ax.axis("off")

    fig.tight_layout()
    return fig

fig = show_raw_repeats(INPUT_TIF, z_indices=range(PLANE_START, min(PLANE_START + 3, N_PLANES)), downsample=DISPLAY_DOWNSAMPLE)
plt.show()


## 6. Run ScanImage alignment / averaging

For validation, keep `PLANE_STOP=3` first. Once the QC looks right, set `PLANE_STOP=None` and rerun for the full volume.


In [ ]:

config = ScanImageAverageConfig(
    input_tif=INPUT_TIF,
    out_dir=OUT_DIR,
    n_planes=N_PLANES,
    repeats_per_plane=REPEATS_PER_PLANE,
    n_channels=N_CHANNELS,
    alignment_channel=ALIGNMENT_CHANNEL,
    order=ORDER,
    plane_start=PLANE_START,
    plane_stop=PLANE_STOP,
    template_method=TEMPLATE_METHOD,
    align=ALIGN,
    registration_binning=REGISTRATION_BINNING,
    upsample_factor=UPSAMPLE_FACTOR,
    max_shift_px=MAX_SHIFT_PX,
    highpass_sigma_px=HIGHPASS_SIGMA_PX,
    interpolation_order=INTERPOLATION_ORDER,
    output_dtype=OUTPUT_DTYPE,
    output_compression=OUTPUT_COMPRESSION,
    write_qc_png=WRITE_QC_PNG,
    bidiphase=BIDIPHASE,
    bidi_line_parity=BIDI_LINE_PARITY,
    bidi_fill_mode=BIDI_FILL_MODE,
    straighten_z=STRAIGHTEN_Z,
    z_anchor=Z_ANCHOR,
    z_template_radius=Z_TEMPLATE_RADIUS,
    z_registration_binning=Z_REGISTRATION_BINNING,
    z_upsample_factor=Z_UPSAMPLE_FACTOR,
    z_max_step_shift_px=Z_MAX_STEP_SHIFT_PX,
    z_highpass_sigma_px=Z_HIGHPASS_SIGMA_PX,
    z_smooth_window=Z_SMOOTH_WINDOW,
)

summary = average_scanimage_volume(config)
summary


## 7. Load summary and shift table

The shift table is one row per `z × repeat`, estimated on the alignment channel and applied to all channels.


In [ ]:

summary_path = Path(summary["summary_json"])
shifts_path = Path(summary["shifts_csv"])

with summary_path.open("r") as f:
    summary_loaded = json.load(f)

shift_df = pd.read_csv(shifts_path)
shift_df["shift_magnitude_px"] = np.sqrt(shift_df["shift_y_px"] ** 2 + shift_df["shift_x_px"] ** 2)

print(f"Summary JSON: {summary_path}")
print(f"Shift CSV: {shifts_path}")
print(f"Rows: {len(shift_df)}")
shift_df.head()


## 8. Shift diagnostics

For a well-behaved structural stack, shifts should usually be small and smoothly distributed. A large number of rejected frames or a heavy-tailed shift distribution suggests the registration settings or stack layout need review.


In [ ]:

display(shift_df.groupby("z_index").agg(
    n=("repeat_index", "count"),
    accepted_frac=("accepted", "mean"),
    median_shift_y_px=("shift_y_px", "median"),
    median_shift_x_px=("shift_x_px", "median"),
    max_shift_magnitude_px=("shift_magnitude_px", "max"),
    mean_error=("error", "mean"),
))

rejected = shift_df.loc[~shift_df["accepted"].astype(bool)]
if len(rejected):
    print("Rejected frames:")
    display(rejected)
else:
    print("No rejected frames.")


In [ ]:

def plot_shift_traces(df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(10, 4))
    for z, g in df.groupby("z_index"):
        ax.plot(g["repeat_index"], g["shift_y_px"], marker="o", linewidth=1, label=f"z {z} y")
        ax.plot(g["repeat_index"], g["shift_x_px"], marker="x", linewidth=1, linestyle="--", label=f"z {z} x")
    ax.axhline(0, linewidth=1)
    ax.set_xlabel("Repeat index")
    ax.set_ylabel("Estimated shift to apply (px)")
    ax.set_title("Rigid x/y shifts by z-plane")
    ax.legend(ncol=2, fontsize=8, frameon=False)
    fig.tight_layout()
    return fig

fig = plot_shift_traces(shift_df)
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(shift_df["shift_magnitude_px"].dropna(), bins=30)
ax.set_xlabel("Shift magnitude (px)")
ax.set_ylabel("Frame count")
ax.set_title("Distribution of estimated shift magnitudes")
fig.tight_layout()
plt.show()


## 9. Inspect output TIFF(s) lazily

This reads only selected planes from the averaged output TIFF, not the whole volume.


In [ ]:

output_paths = {k: Path(v) for k, v in summary_loaded["outputs"].items()}
output_paths


### Optional z-straightening outputs

If `STRAIGHTEN_Z=True`, inspect the straightened outputs and z-shift CSV here. You can also run `straighten_volume_tiff()` posthoc on an existing averaged volume.

In [ ]:

straightened_output_paths = {k: Path(v) for k, v in summary_loaded.get("straightened_outputs", {}).items() if v}
z_straightening_csv = summary_loaded.get("z_straightening_csv")

print("Straightened outputs:")
display(straightened_output_paths)
print("Z-straightening CSV:", z_straightening_csv)

if z_straightening_csv:
    z_shift_df = pd.read_csv(z_straightening_csv)
    z_shift_df["smooth_shift_magnitude_px"] = np.sqrt(
        z_shift_df["smooth_shift_y_px"] ** 2 + z_shift_df["smooth_shift_x_px"] ** 2
    )
    display(z_shift_df.head())
else:
    z_shift_df = None


In [ ]:

if z_shift_df is not None:
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    axes[0].plot(z_shift_df["z_index"], z_shift_df["raw_shift_y_px"], ".", alpha=0.35, label="raw y")
    axes[0].plot(z_shift_df["z_index"], z_shift_df["smooth_shift_y_px"], "-", label="smoothed y")
    axes[0].set_ylabel("Y shift (px)")
    axes[0].legend()

    axes[1].plot(z_shift_df["z_index"], z_shift_df["raw_shift_x_px"], ".", alpha=0.35, label="raw x")
    axes[1].plot(z_shift_df["z_index"], z_shift_df["smooth_shift_x_px"], "-", label="smoothed x")
    axes[1].set_ylabel("X shift (px)")
    axes[1].legend()

    axes[2].plot(z_shift_df["z_index"], z_shift_df["smooth_shift_magnitude_px"], "-", label="magnitude")
    axes[2].set_ylabel("Magnitude (px)")
    axes[2].set_xlabel("z index")
    axes[2].legend()

    anchor = int(z_shift_df["anchor_z_index"].iloc[0])
    for ax in axes:
        ax.axvline(anchor, color="k", linestyle="--", alpha=0.4)
        ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    print("No z-straightening CSV available. Set STRAIGHTEN_Z=True or run the posthoc cell below.")


### Posthoc z-straightening of an existing averaged TIFF

Use this if you already produced `*_avg_ch1.tif` and want to create `*_straightened.tif` without rerunning repeat-level averaging.

In [ ]:

POSTHOC_STRAIGHTEN = False

if POSTHOC_STRAIGHTEN:
    output_ch_key = f"channel_{ALIGNMENT_CHANNEL + 1}"
    avg_tif = output_paths[output_ch_key]
    posthoc_out = avg_tif.with_name(f"{avg_tif.stem}_straightened.tif")
    posthoc_summary = straighten_volume_tiff(
        avg_tif,
        posthoc_out,
        anchor_z=Z_ANCHOR,
        template_radius=Z_TEMPLATE_RADIUS,
        binning=Z_REGISTRATION_BINNING,
        upsample_factor=Z_UPSAMPLE_FACTOR,
        max_step_shift_px=Z_MAX_STEP_SHIFT_PX,
        highpass_sigma_px=Z_HIGHPASS_SIGMA_PX,
        smooth_window=Z_SMOOTH_WINDOW,
        output_dtype=OUTPUT_DTYPE,
        output_compression=OUTPUT_COMPRESSION,
    )
    display(posthoc_summary)
else:
    print("Set POSTHOC_STRAIGHTEN=True to straighten an existing averaged TIFF.")


In [ ]:

def inspect_tiff(path: Path):
    with tifffile.TiffFile(path) as tif:
        print(path)
        print(f"n_pages: {len(tif.pages)}")
        print(f"series count: {len(tif.series)}")
        for i, s in enumerate(tif.series):
            print(f"  series {i}: shape={s.shape}, dtype={s.dtype}, axes={s.axes}")
        print(f"page[0].shape: {tif.pages[0].shape}")
        print(f"page[0].dtype: {tif.pages[0].dtype}")
        desc = tif.pages[0].description or ""
        print("description excerpt:")
        print(desc[:500])

for path in output_paths.values():
    inspect_tiff(path)


In [ ]:

def read_output_planes(path: Path, z_indices, downsample: int = 1) -> dict[int, np.ndarray]:
    """Read selected z-planes from a ZYX output TIFF."""
    out = {}
    with tifffile.TiffFile(path) as tif:
        for z in z_indices:
            img = tif.pages[int(z)].asarray()
            if downsample > 1:
                img = img[::downsample, ::downsample]
            out[int(z)] = img
    return out


def show_output_montage(path: Path, z_indices=None, downsample=8):
    with tifffile.TiffFile(path) as tif:
        n_pages = len(tif.pages)
    if z_indices is None:
        z_indices = np.linspace(0, n_pages - 1, min(12, n_pages), dtype=int)

    planes = read_output_planes(path, z_indices, downsample=downsample)
    n = len(planes)
    ncols = min(6, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.7 * ncols, 2.7 * nrows), squeeze=False)
    axes = axes.ravel()

    for ax, (z, img) in zip(axes, planes.items()):
        lo, hi = robust_limits(img)
        ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(f"z={z}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(planes):]:
        ax.axis("off")

    fig.suptitle(path.name, y=1.02)
    fig.tight_layout()
    return fig

for label, path in output_paths.items():
    fig = show_output_montage(path, downsample=DISPLAY_DOWNSAMPLE)
    plt.show()


## 10. Output intensity / focus diagnostics across z

This scans the output TIFF plane-by-plane and computes simple summary metrics. For the full 2048 × 2048 × 177 output, this reads the averaged output volume from disk once, but still does not load it all into memory simultaneously.


In [ ]:

def compute_output_z_metrics(path: Path, step: int = 1, downsample: int = 4) -> pd.DataFrame:
    rows = []
    with tifffile.TiffFile(path) as tif:
        for z in range(0, len(tif.pages), step):
            img = tif.pages[z].asarray()
            if downsample > 1:
                img = img[::downsample, ::downsample]
            finite = np.isfinite(img)
            if finite.any():
                vals = img[finite]
                rows.append({
                    "z_index": z,
                    "mean": float(np.mean(vals)),
                    "median": float(np.median(vals)),
                    "std": float(np.std(vals)),
                    "p99": float(np.percentile(vals, 99)),
                    "p99_8": float(np.percentile(vals, 99.8)),
                    "finite_fraction": float(finite.mean()),
                })
            else:
                rows.append({
                    "z_index": z,
                    "mean": np.nan,
                    "median": np.nan,
                    "std": np.nan,
                    "p99": np.nan,
                    "p99_8": np.nan,
                    "finite_fraction": 0.0,
                })
    return pd.DataFrame(rows)

metrics_by_channel = {}
for label, path in output_paths.items():
    metrics_by_channel[label] = compute_output_z_metrics(
        path, step=Z_PROFILE_STEP, downsample=DISPLAY_DOWNSAMPLE
    )
    print(label, path)
    display(metrics_by_channel[label].head())


In [ ]:

for label, metrics in metrics_by_channel.items():
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(metrics["z_index"], metrics["mean"], marker="o", linewidth=1, label="mean")
    ax.plot(metrics["z_index"], metrics["p99_8"], marker="o", linewidth=1, label="p99.8")
    ax.set_xlabel("z index")
    ax.set_ylabel("Intensity")
    ax.set_title(f"Output z-profile: {label}")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()


## 11. Raw repeat average versus aligned average for one z-plane

This is a useful local check. It reads all raw repeats for one selected z-plane, computes an unregistered mean, then compares it to the saved registered average.


In [ ]:

raw_z = SELECTED_Z_FOR_RAW_CHECK
output_ch_key = f"channel_{ALIGNMENT_CHANNEL + 1}"
output_path = output_paths[output_ch_key]

with tifffile.TiffFile(INPUT_TIF) as raw_tif:
    raw_frames = read_plane_channel_frames(
        raw_tif, z_index=raw_z, channel_index=ALIGNMENT_CHANNEL, spec=spec
    )

raw_mean = np.mean(np.stack([f.astype(np.float32, copy=False) for f in raw_frames], axis=0), axis=0)

# Recompute shifts for this plane using current notebook settings, then compute an aligned mean.
template = make_template(raw_frames, method=TEMPLATE_METHOD)
recomputed_shifts = []
for frame in raw_frames:
    result = estimate_rigid_shift(
        template,
        frame,
        upsample_factor=UPSAMPLE_FACTOR,
        max_shift_px=MAX_SHIFT_PX,
        binning=REGISTRATION_BINNING,
        highpass_sigma_px=HIGHPASS_SIGMA_PX,
    )
    recomputed_shifts.append(result.shift_yx if result.accepted else (0.0, 0.0))

aligned_mean_recomputed = mean_shifted_frames(
    raw_frames,
    recomputed_shifts,
    order=INTERPOLATION_ORDER,
)

with tifffile.TiffFile(output_path) as out_tif:
    saved_aligned_mean = out_tif.pages[raw_z - PLANE_START].asarray()

print("Recomputed shifts:")
print(pd.DataFrame(recomputed_shifts, columns=["shift_y_px", "shift_x_px"]).describe())


In [ ]:

def show_comparison_images(images: dict[str, np.ndarray], downsample=8):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), squeeze=False)
    axes = axes.ravel()
    for ax, (title, img) in zip(axes, images.items()):
        thumb = img[::downsample, ::downsample]
        lo, hi = robust_limits(thumb)
        ax.imshow(thumb, cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    return fig

fig = show_comparison_images(
    {
        "raw mean": raw_mean,
        "aligned mean recomputed": aligned_mean_recomputed,
        "saved aligned mean": saved_aligned_mean,
        "saved - raw": saved_aligned_mean - raw_mean,
    },
    downsample=DISPLAY_DOWNSAMPLE,
)
plt.show()

print("Max abs difference between saved and recomputed aligned mean:", np.nanmax(np.abs(saved_aligned_mean - aligned_mean_recomputed)))


## 12. Save notebook-generated diagnostic figures/tables

This writes a compact diagnostics folder next to the averaged TIFF output.


In [ ]:

DIAG_DIR = OUT_DIR / "notebook_diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

# Save shifts with magnitude added.
shift_df.to_csv(DIAG_DIR / "alignment_shifts_with_magnitude.csv", index=False)

# Save z metrics.
for label, metrics in metrics_by_channel.items():
    metrics.to_csv(DIAG_DIR / f"{label}_z_metrics.csv", index=False)

# Save a quick shift diagnostic plot.
fig = plot_shift_traces(shift_df)
fig.savefig(DIAG_DIR / "shift_traces.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# Save output montage(s).
for label, path in output_paths.items():
    fig = show_output_montage(path, downsample=DISPLAY_DOWNSAMPLE)
    fig.savefig(DIAG_DIR / f"{label}_output_montage.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Wrote diagnostics to: {DIAG_DIR}")


## 13. Optional: open the averaged volume in napari

This is optional and intended for local interactive QC. It will try to load the full averaged TIFF into memory, so only use it when the output size is manageable on your workstation.


In [ ]:

OPEN_NAPARI = False

if OPEN_NAPARI:
    import napari
    viewer = napari.Viewer()
    for label, path in output_paths.items():
        data = tifffile.imread(path)
        viewer.add_image(data, name=label, scale=(1, 1, 1))
    napari.run()
else:
    print("Set OPEN_NAPARI=True to launch napari locally.")


## 14. Full-run checklist

Before processing the full 177-plane stack:

1. Confirm the raw contact sheet shows repeated views within each z-plane.
2. Confirm `read_tiff_stack_spec` accounts for all pages.
3. Run `PLANE_START=0`, `PLANE_STOP=3` and inspect:
   - output montage
   - shift magnitudes
   - rejected frames
   - raw mean versus aligned mean
4. Then set `PLANE_STOP=None` and rerun.
5. If shifts are nearly all zero, that is acceptable; the code still verifies each plane and applies the same transformation to future channels.
6. If many frames are rejected, first check `ORDER`, `N_CHANNELS`, and `REPEATS_PER_PLANE`; then tune `REGISTRATION_BINNING`, `HIGHPASS_SIGMA_PX`, and `MAX_SHIFT_PX`.
